# Notebook 05 — MFTMA Manifold Certification (All Ablation Conditions)

**Method:** Replica Mean-Field Theory of Manifold Analysis (MFTMA)  
**Ablation conditions:** B7 · B8 · B9 · B13 · A2 (blur) · A4 (trace-based)  
**New module:** `src/mftma.py` — local simplified MFTMA + optional neural_manifolds hook  
**Outputs:** `results/05_manifold_stats.json` · `results/05_manifold_geometry.png`

---

## Purpose

MFTMA (Chung et al. 2018, Dapello & Chung 2021) certifies whether the representation
of object identity **manifolds** in a neural network's feature space is linearly
separable. Three key geometric quantities determine classification capacity:

| Metric | Symbol | Meaning |
|--------|--------|---------|
| Manifold capacity | $\alpha_c$ | Max number of random dichotomies solvable by a linear classifier |
| Manifold radius | $R_M$ | RMS distance from manifold centre (object variability) |
| Manifold dimension | $D_M$ | Intrinsic dimensionality of within-class variation |

Hypothesis: adding foveation + V1 noise + trace-based periphery should
$\alpha_c\uparrow$, $R_M\downarrow$, $D_M\downarrow$ — more compact, linearly separable manifolds.

## Mathematics

For $P$ object manifolds each with $M$ exemplars in $\mathbb{R}^N$ (feature dimension):

**Manifold capacity** $\alpha_c = P / N$ at the critical load (linear separability transition):

$$
\alpha_c^{-1} \approx \int_0^\infty Dt\,(D_M \cdot t^2 + R_M^2 \cdot t^4)
$$

(mean-field approximation; full solution from replica method).

**Manifold radius:**

$$
R_M = \frac{1}{P}\sum_{p=1}^P \sqrt{\frac{1}{M}\sum_{m=1}^M \lVert x_{pm} - \mu_p\rVert^2}
$$

**Manifold dimension** (participation ratio):

$$
D_M = \frac{\left(\sum_k \lambda_k\right)^2}{\sum_k \lambda_k^2},\quad
\{\lambda_k\} = \text{within-class PCA eigenvalues}
$$

**Local implementation note:** Full MFTMA uses the `neural_manifolds_replicaMFT`
package (`pip install neural-manifolds`). If unavailable, this notebook falls back to a
sklearn approximation (linear SVM capacity, PCA radius/dimension).


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else: PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path: sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import common, models
from src.foveation import FoveatedTransform, TraceBasedPeriphery, FlatBlurPeriphery, SNRProfile, build_foveated_transform

CFG = {
    "seed": 42, "smoke_test": True,
    "image_size": 32, "n_classes": 10, "ppd": 4.0,
    "fovea_deg": 1.0, "transition_deg": 0.5,
    "rblur_sigma0": 0.5, "rblur_slope": 1.5,
    "snr0_db": 30.0, "beta": 2.0, "patch_size": 8,
    # MFTMA
    "n_manifold_images":  64,   # images per class (smoke: 8 or 64)
    "n_classes_mftma":    5,    # classes to analyse
    "n_dichotomies":      20,
    "pca_variance":       0.95,
    # Conditions to analyse
    "conditions": ["B7_resnet50", "B8_vone_off", "B9_vone_on",
                   "A2_rblur_periphery", "A4_trace_periphery"],
    "result_dir": os.path.join(PROJECT_ROOT, "results"),
    "data_dir":   os.path.join(PROJECT_ROOT, "data"),
}

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
print(f"Device: {device} | n_manifold_images={CFG['n_manifold_images']}")


In [ ]:
# ── Simplified MFTMA functions ────────────────────────────────────────────

def manifold_radius(activations_by_class):
    """R_M = mean of per-class RMS distance from class centroid.
    activations_by_class: list of [M, d] arrays (one per class)."""
    radii = []
    for acts in activations_by_class:
        mu = acts.mean(0, keepdims=True)
        radii.append(np.sqrt(((acts - mu) ** 2).sum(1).mean()))
    return float(np.mean(radii))


def manifold_dimension(activations_by_class, variance_threshold=0.95):
    """D_M via participation ratio of within-class PCA eigenvalues."""
    dims = []
    for acts in activations_by_class:
        if acts.shape[0] < 2: continue
        mu = acts.mean(0)
        centered = acts - mu
        cov = (centered.T @ centered) / max(len(acts) - 1, 1)
        eigs = np.linalg.eigvalsh(cov)
        eigs = np.sort(eigs[::-1])
        eigs = eigs[eigs > 0]
        if len(eigs) == 0: continue
        pr = (eigs.sum() ** 2) / (eigs ** 2).sum()   # participation ratio
        dims.append(float(pr))
    return float(np.mean(dims)) if dims else float("nan")


def manifold_capacity_approx(activations_by_class, n_dichotomies=20, seed=0):
    """Approximate alpha_c via random dichotomy test with LinearSVC.

    For each of n_dichotomies random binary assignments of class labels:
      - Train LinearSVC on 80% of data
      - Test on remaining 20%
    alpha_c approx = fraction of dichotomies with test accuracy >= 0.95
    """
    rng = np.random.RandomState(seed)
    n_classes = len(activations_by_class)
    all_feats  = np.concatenate(activations_by_class, axis=0)  # [N*M, d]
    class_ids  = np.concatenate([np.full(len(a), i)
                                  for i, a in enumerate(activations_by_class)])

    scaler = StandardScaler()
    all_feats_s = scaler.fit_transform(all_feats)
    n_total = len(all_feats_s)
    split = int(0.8 * n_total)

    successes = 0
    for _ in range(n_dichotomies):
        binary_labels = rng.choice([0, 1], size=n_classes)
        y_bin = binary_labels[class_ids]
        perm  = rng.permutation(n_total)
        X_tr, y_tr = all_feats_s[perm[:split]],  y_bin[perm[:split]]
        X_te, y_te = all_feats_s[perm[split:]],  y_bin[perm[split:]]
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
            continue
        try:
            clf = LinearSVC(max_iter=200, tol=1e-3)
            clf.fit(X_tr, y_tr)
            acc = (clf.predict(X_te) == y_te).mean()
            if acc >= 0.95: successes += 1
        except Exception:
            pass
    return float(successes / n_dichotomies)


def compute_manifold_stats(activations_by_class, n_dichotomies=20, seed=0):
    """Compute all three MFTMA metrics."""
    return {
        "alpha_c": manifold_capacity_approx(activations_by_class, n_dichotomies, seed),
        "R_M":     manifold_radius(activations_by_class),
        "D_M":     manifold_dimension(activations_by_class),
    }


print("MFTMA helper functions defined.")
print("Note: Using sklearn-based approximation. Full MFTMA: pip install neural-manifolds")


In [ ]:
# ── Build evaluation conditions ───────────────────────────────────────────

PRETRAINED = not CFG["smoke_test"]

snr_prof = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"], ppd=CFG["ppd"])

# Map condition name -> (model, norm, foveation_transform_or_None)
conditions = {}

# B7: ResNet-50, no foveation
m_b7, n_b7 = models.build_resnet50(pretrained=PRETRAINED)
conditions["B7_resnet50"] = (m_b7.eval(), n_b7, None)

# B8: VOneBlock noise off
from src.overrides import apply_vone_block_input_gradient_fix, set_v1_noise_mode
apply_vone_block_input_gradient_fix()
m_b8, n_b8 = models.build_voneresnet50(pretrained=PRETRAINED)
set_v1_noise_mode(m_b8, mode=None)
conditions["B8_vone_off"] = (m_b8.eval(), n_b8, None)

# B9: VOneBlock noise on
m_b9, n_b9 = models.build_voneresnet50(pretrained=PRETRAINED)
set_v1_noise_mode(m_b9, mode="neuronal")
conditions["B9_vone_on"] = (m_b9.eval(), n_b9, None)

# A2: R-Blur flat-blur periphery + ResNet-50
m_a2, n_a2 = models.build_resnet50(pretrained=PRETRAINED)
ft_a2 = build_foveated_transform(
    "blur", snr_prof, CFG["patch_size"], CFG["fovea_deg"],
    CFG["transition_deg"], CFG["ppd"], (0.5, 0.5),
    CFG["rblur_sigma0"], CFG["rblur_slope"])
conditions["A2_rblur_periphery"] = (m_a2.eval(), n_a2, ft_a2)

# A4: Trace-based periphery + ResNet-50
m_a4, n_a4 = models.build_resnet50(pretrained=PRETRAINED)
ft_a4 = build_foveated_transform(
    "trace", snr_prof, CFG["patch_size"], CFG["fovea_deg"],
    CFG["transition_deg"], CFG["ppd"], (0.5, 0.5))
conditions["A4_trace_periphery"] = (m_a4.eval(), n_a4, ft_a4)

print(f"Built {len(conditions)} conditions: {list(conditions.keys())}")


In [ ]:
# ── Extract activations per class ─────────────────────────────────────────

MU_T  = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD_T = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def make_proxy_images_by_class(n_classes, n_per_class, image_size, seed=42):
    """Synthetic class-structured images for smoke_test."""
    rng = np.random.RandomState(seed)
    images_by_class = []
    for c in range(n_classes):
        # Each class has a distinctive colour bias
        bias = torch.tensor(rng.uniform(0, 1, 3), dtype=torch.float32).view(3, 1, 1)
        imgs = torch.rand(n_per_class, 3, image_size, image_size) * 0.3 + bias * 0.7
        images_by_class.append(imgs.clamp(0.0, 1.0))
    return images_by_class   # list of [M, 3, H, W] raw [0,1]


def extract_penultimate(model, norm, fov_transform, images_by_class, device_):
    """Extract penultimate (pre-logit) features for each class.
    Returns list of [M, d] numpy arrays."""
    base = models.unwrap_model(model)
    # Register hook at avgpool-like layer
    feats_bucket = []

    def _hook(_m, _in, out):
        feats_bucket.append(out.detach().cpu().flatten(1))

    # Find a good hook point
    hook_mod = None
    if hasattr(base, 'avgpool'):                  # ResNet
        hook_mod = base.avgpool
    elif hasattr(base, 'model') and hasattr(base.model, 'avgpool'):  # VOneNet
        hook_mod = base.model.avgpool
    elif hasattr(base, 'IT'):                     # CORnet-S
        hook_mod = base.IT
    else:
        hook_mod = list(base.children())[-2]      # second-to-last module

    handle = hook_mod.register_forward_hook(_hook)

    norm_model = models.NormalizedModel(model, norm).to(device_).eval()

    result = []
    for class_imgs in images_by_class:
        feats_bucket.clear()
        with torch.no_grad():
            for i in range(0, len(class_imgs), 16):
                batch = class_imgs[i:i+16].to(device_)
                if fov_transform is not None:
                    batch_fov = torch.stack([fov_transform(batch[j])
                                             for j in range(batch.size(0))])
                    norm_model(batch_fov)
                else:
                    norm_model(batch)
        class_feats = torch.cat(feats_bucket, dim=0).numpy()
        result.append(class_feats)

    handle.remove()
    return result


# Generate proxy images
N_CLS = CFG["n_classes_mftma"]
N_PER = CFG["n_manifold_images"]
images_by_class = make_proxy_images_by_class(N_CLS, N_PER, CFG["image_size"])
print(f"Proxy data: {N_CLS} classes x {N_PER} images = {N_CLS*N_PER} total")

# Extract activations for all conditions
print("\nExtracting activations...")
activations = {}
for cond_name in CFG["conditions"]:
    if cond_name not in conditions:
        print(f"  {cond_name}: skipped (not built)")
        continue
    m, n, ft = conditions[cond_name]
    acts = extract_penultimate(m, n, ft, images_by_class, device)
    activations[cond_name] = acts
    feat_dim = acts[0].shape[1] if acts else 0
    print(f"  {cond_name}: {N_CLS} classes x {N_PER} x {feat_dim}")


In [ ]:
# ── Compute MFTMA metrics ─────────────────────────────────────────────────

print("Computing manifold geometry statistics...")
print(f"{'Condition':25s}  {'alpha_c':>8}  {'R_M':>8}  {'D_M':>8}")
print("-" * 56)

manifold_stats = {}
for cond_name, acts in activations.items():
    stats = compute_manifold_stats(acts, n_dichotomies=CFG["n_dichotomies"],
                                    seed=CFG["seed"])
    manifold_stats[cond_name] = stats
    print(f"  {cond_name:25s}  {stats['alpha_c']:8.4f}  "
          f"{stats['R_M']:8.4f}  {stats['D_M']:8.4f}")

print("\nNote: alpha_c via random dichotomy test (LinearSVC approx).")
print("For full MFTMA: import mftma; mftma.manifold_analysis_corr(...)")


In [ ]:
# ── Manifold geometry visualisation ─────────────────────────────────────

if not manifold_stats:
    print("No stats to plot.")
else:
    cond_names = list(manifold_stats.keys())
    alphas = [manifold_stats[c]["alpha_c"] for c in cond_names]
    radii  = [manifold_stats[c]["R_M"]     for c in cond_names]
    dims   = [manifold_stats[c]["D_M"]     for c in cond_names]
    colors = plt.cm.Set2(np.linspace(0, 1, len(cond_names)))
    short  = [c.split("_")[0] for c in cond_names]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, vals, ylabel, title in zip(
        axes,
        [alphas, radii, dims],
        ["alpha_c (capacity, higher=better)",
         "R_M (radius, lower=better)",
         "D_M (dimension, lower=better)"],
        ["Manifold Capacity alpha_c",
         "Manifold Radius R_M",
         "Manifold Dimension D_M"],
    ):
        bars = ax.bar(range(len(cond_names)), vals, color=colors)
        ax.set_xticks(range(len(cond_names)))
        ax.set_xticklabels(short, rotation=20, ha="right")
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.grid(axis="y", alpha=0.3)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.002 * max(vals),
                        f"{val:.3f}", ha="center", va="bottom", fontsize=8)

    fig.suptitle(
        "MFTMA-proxy: manifold geometry across ablation conditions\n"
        "(sklearn approximation — synthetic data, not real neural responses)",
        fontsize=9, y=1.02,
    )
    plt.tight_layout()
    geo_path = os.path.join(CFG["result_dir"], "05_manifold_geometry.png")
    common.save_figure(fig, geo_path, dpi=130)
    plt.close(fig)
    print(f"Saved: {geo_path}")


In [ ]:
_mftma_path = os.path.join(PROJECT_ROOT, 'src', 'mftma.py')
_mftma_content = '"""\nMFTMA manifold geometry analysis for the foveated-vision thesis.\n\nAttempts to use the official neural_manifolds_replicaMFT package if available;\notherwise falls back to a sklearn-based approximation.\n\nPublic API\n----------\nmanifold_radius          -- R_M: RMS distance from class centroid\nmanifold_dimension       -- D_M: participation ratio of within-class PCA eigs\nmanifold_capacity_approx -- alpha_c: random dichotomy LinearSVC fraction\ncompute_manifold_stats   -- all three metrics in one call\n"""\n\nimport numpy as np\nfrom sklearn.svm import LinearSVC\nfrom sklearn.preprocessing import StandardScaler\n\ntry:\n    from mftma import manifold_analysis_corr as _mftma_official\n    HAS_MFTMA = True\nexcept ImportError:\n    HAS_MFTMA = False\n\n\ndef manifold_radius(activations_by_class):\n    """R_M: mean per-class RMS distance from centroid."""\n    radii = []\n    for acts in activations_by_class:\n        mu = acts.mean(0, keepdims=True)\n        radii.append(float(np.sqrt(((acts - mu)**2).sum(1).mean())))\n    return float(np.mean(radii)) if radii else float(\'nan\')\n\n\ndef manifold_dimension(activations_by_class):\n    """D_M: mean participation ratio of within-class PCA eigenvalues."""\n    dims = []\n    for acts in activations_by_class:\n        if acts.shape[0] < 2: continue\n        c = acts - acts.mean(0)\n        eigs = np.linalg.eigvalsh(c.T @ c / max(len(acts)-1, 1))\n        eigs = eigs[eigs > 0]\n        if len(eigs) == 0: continue\n        dims.append(float((eigs.sum()**2) / (eigs**2).sum()))\n    return float(np.mean(dims)) if dims else float(\'nan\')\n\n\ndef manifold_capacity_approx(activations_by_class, n_dichotomies=20, seed=0):\n    """alpha_c: fraction of random dichotomies solved by LinearSVC."""\n    rng = np.random.RandomState(seed)\n    n_cls = len(activations_by_class)\n    feats = np.concatenate(activations_by_class, axis=0)\n    ids   = np.concatenate([np.full(len(a), i) for i, a in enumerate(activations_by_class)])\n    scaler = StandardScaler()\n    feats_s = scaler.fit_transform(feats)\n    split = int(0.8 * len(feats_s))\n    ok = 0\n    for _ in range(n_dichotomies):\n        y = rng.choice([0, 1], size=n_cls)[ids]\n        p = rng.permutation(len(feats_s))\n        X_tr, y_tr = feats_s[p[:split]], y[p[:split]]\n        X_te, y_te = feats_s[p[split:]], y[p[split:]]\n        if len(set(y_tr)) < 2: continue\n        try:\n            clf = LinearSVC(max_iter=200, tol=1e-3)\n            clf.fit(X_tr, y_tr)\n            if (clf.predict(X_te) == y_te).mean() >= 0.95: ok += 1\n        except Exception: pass\n    return float(ok / n_dichotomies)\n\n\ndef compute_manifold_stats(activations_by_class, n_dichotomies=20, seed=0):\n    """Compute all MFTMA metrics. activations_by_class: list of [M,d] arrays."""\n    if HAS_MFTMA:\n        # Use official neural_manifolds_replicaMFT\n        kappa, alpha_c, R_M, D_M, *_ = _mftma_official(\n            [a.T for a in activations_by_class], kappa=0.0, n_t=200)\n        return {\'alpha_c\': float(alpha_c), \'R_M\': float(R_M), \'D_M\': float(D_M)}\n    return {\n        \'alpha_c\': manifold_capacity_approx(activations_by_class, n_dichotomies, seed),\n        \'R_M\':     manifold_radius(activations_by_class),\n        \'D_M\':     manifold_dimension(activations_by_class),\n    }'
with open(_mftma_path, 'w') as _f: _f.write(_mftma_content)
print(f'Written: {_mftma_path}')
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('mftma', _mftma_path)
_m = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_m)
assert hasattr(_m, 'compute_manifold_stats')
print('src/mftma.py verified: compute_manifold_stats present')
print(f'  HAS_MFTMA (neural_manifolds_replicaMFT): {_m.HAS_MFTMA}')

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────
output = {
    "notebook": "05_mftma_certification",
    "cfg": {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "manifold_stats": manifold_stats,
    "note": ("smoke_test=True: synthetic proxy data + random models. "
             "Stats reflect pipeline, not real neural geometry."),
}
rpath = os.path.join(CFG["result_dir"], "05_manifold_stats.json")
common.save_json(output, rpath)
print(f"Saved: {rpath}")
print(f"05_manifold_geometry.png : {os.path.exists(os.path.join(CFG['result_dir'], '05_manifold_geometry.png'))}")
print(f"src/mftma.py             : {os.path.exists(os.path.join(PROJECT_ROOT, 'src', 'mftma.py'))}")
print("\n── Notebook 05 complete ✓")
